In [26]:
from langgraph.graph import StateGraph, START, END  
# StateGraph : 그래프는 상태를 가진다는 것 (state 상태란 그래프를 통해 이동하는 데이터임)
# START, END는 예약어 같은거임 
from langgraph.pregel.main import Input
from typing_extensions import TypedDict 

class PrivateState(TypedDict): # node 내부적으로 사용되는 state 
    a : int
    b : int 

class InputState(TypedDict): #사용자로부터 오는 것 (사용자 제공 state)
    hello : str

class OutputState(TypedDict): # agent의 output 임 
    bye : str

class MegaPrivate(TypedDict): # 아래 StateGraph에 추가하지 않아도 호출 가능 
    secret : bool

graph_builder=StateGraph(
    PrivateState ,
    input_schema=InputState,
    output_schema=OutputState,
)


In [ ]:
# 노드는 그 자리에서 state를 받을 수 있음. (당연한말)
# 아래 노드 생성(이게 우리 작업 단위임)

 # node_one은 inputState를 받고 InputState형태를 return한다는 의미 (-> InputState 는 안적어도 되지만 적으면 좋음 )
def node_one(state: InputState) -> InputState:
    print("node_one ->", state)
    return {"hello": "world"}


def node_two(state: PrivateState) -> PrivateState:
    print("node_two ->", state)
    return {
        "a": 1,
    }


def node_three(state: PrivateState) -> PrivateState:
    print("node_three ->", state)
    return {"b": 1}
 
# privateState를 받고 OutputState를 return 해준다는 뜻 
def node_four(state: PrivateState) -> OutputState:
    print("node_four ->", state)
    return {
        "bye": "world",
    }


def node_five(state: OutputState):
    return {"secret": True}


def node_six(state: MegaPrivate): # MegaPrivate 는 five,, six 처럼 접근 가능 five에서 고쳤더라도 여기서 호출안하면 안보여짐. MegaPrivate 를 호출해야함함
    print(state)

In [28]:
# 우리는 위에 만든 node를 그래프에게 줄것임(graph_builder)

graph_builder.add_node("node_one", node_one)  # node_one이라는 이름으로 node_one함수 실행  
graph_builder.add_node("node_two", node_two) 
graph_builder.add_node("node_three", node_three) 
graph_builder.add_node("node_four", node_four) 
graph_builder.add_node("node_five", node_five) 
graph_builder.add_node("node_six", node_six) 

# node들 끼리 연결하기 위해 edge를 만든다(edge는 화살표) 
graph_builder.add_edge(START, "node_one") # start에서 "node_one"으로 화살표 추가한다는 뜻
graph_builder.add_edge("node_one", "node_two")
graph_builder.add_edge("node_two", "node_three")
graph_builder.add_edge("node_three", "node_four")
graph_builder.add_edge("node_four", "node_five")
graph_builder.add_edge("node_five", "node_six")
graph_builder.add_edge("node_six", END)
 


In [29]:
# graph = graph_builder.compile()

# graph

In [ ]:
graph = graph_builder.compile() 

graph.invoke(
    {"hello": "world"},
)

node_one -> {'hello': 'world'}
node_two -> {}
node_three -> {'a': 1}
node_four -> {'a': 1, 'b': 1}
{'secret': True}


{'bye': 'world'}